# 01 · Exploración de datos — ROGII Wellbore Geology Prediction

Objetivo de este notebook: entender **qué datos tenemos y cómo se comportan**, sin construir ningún modelo.

Contexto de la tarea (del pptx incluido en los datos):

- Cada pozo horizontal viene con dos archivos: `<id>__horizontal_well.csv` (trayectoria + gamma ray + target) y `<id>__typewell.csv` (pozo vertical de referencia: TVT, GR y nombre de la capa geológica).
- **TVT** (*True Vertical Thickness*) es la "profundidad geológica" del punto del pozo horizontal expresada en la escala del typewell. Es el target.
- TVT se conoce hasta el **Prediction Start (PS)**; en `TVT_input` está el target hasta el PS y NaN después. Hay que predecir TVT después del PS a partir de XYZ, GR del pozo horizontal y (TVT, GR) del typewell.
- Métrica: RMSE sobre todos los puntos predichos (paso de 1 ft).
- Unidades: pies.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

RAW = Path("../data/raw")
TRAIN = RAW / "train"
TEST = RAW / "test"
TOPS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]  # topes de formación (en Z) del pozo horizontal

## 1. Inventario de archivos

In [ ]:
def list_wells(folder: Path) -> list[str]:
    return sorted(p.name.split("__")[0] for p in folder.glob("*__horizontal_well.csv"))

train_ids = list_wells(TRAIN)
test_ids = list_wells(TEST)

inv = pd.DataFrame({
    "carpeta": ["train", "test"],
    "pozos (horizontal_well.csv)": [len(train_ids), len(test_ids)],
    "typewell.csv": [len(list(TRAIN.glob("*__typewell.csv"))), len(list(TEST.glob("*__typewell.csv")))],
    "png": [len(list(TRAIN.glob("*.png"))), len(list(TEST.glob("*.png")))],
})
display(inv)

print("Otros archivos en raw:", [p.name for p in RAW.iterdir() if p.is_file()])
print("¿Los ids de test también están en train?", set(test_ids) <= set(train_ids), "→", test_ids)

Los 3 pozos de `test/` son **copias de pozos de train sin el target** (mismos ids). El test real de la competencia se evaluaba en Kaggle con pozos ocultos; para una implementación local tendremos que armar nuestra propia validación separando pozos de train.

In [ ]:
sub = pd.read_csv(RAW / "sample_submission.csv")
sub[["well", "idx"]] = sub["id"].str.split("_", expand=True)
sub["idx"] = sub["idx"].astype(int)
display(sub.head(3))
display(sub.groupby("well")["idx"].agg(filas="size", idx_min="min", idx_max="max"))

## 2. Anatomía de un pozo

In [ ]:
def load_well(well_id: str, folder: Path = TRAIN) -> tuple[pd.DataFrame, pd.DataFrame]:
    h = pd.read_csv(folder / f"{well_id}__horizontal_well.csv")
    t = pd.read_csv(folder / f"{well_id}__typewell.csv")
    return h, t

wid = test_ids[0]
h, t = load_well(wid)
ps_idx = int(h["TVT_input"].notna().sum())  # primer índice sin TVT_input = Prediction Start

print(f"Pozo {wid}: {len(h)} filas, PS en fila {ps_idx} (MD={h.MD[ps_idx]:.0f} ft)")
display(h.head(3))
display(h.iloc[ps_idx - 1: ps_idx + 2])
print("Typewell:", t.shape)
display(t.head(3))
display(t.dropna(subset=["Geology"]).groupby("Geology", sort=False)["TVT"].agg(["min", "max", "size"]))

Confirmación con `sample_submission`: el índice de la primera fila a predecir del pozo coincide con el PS (número de filas con `TVT_input` no nulo). Es decir, el `id` de la submission es `<well>_<índice de fila>`.

In [ ]:
test_h, _ = load_well(wid, TEST)
print("Columnas test:", list(test_h.columns))
print("PS (train) =", ps_idx, "| primer idx en sample_submission =", sub.loc[sub.well == wid, "idx"].min())

In [ ]:
def plot_well(h: pd.DataFrame, t: pd.DataFrame, wid: str):
    ps = int(h["TVT_input"].notna().sum())
    md_ps = h.MD[ps]
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, width_ratios=[3, 1.2])

    ax = fig.add_subplot(gs[0, 0])
    ax.plot(h.MD, h.GR, lw=0.6, color="k")
    ax.axvline(md_ps, color="red", ls="--", label="Prediction Start")
    ax.set_ylabel("GR"); ax.set_title("Gamma ray a lo largo del pozo"); ax.legend()

    ax = fig.add_subplot(gs[1, 0], sharex=ax)
    ax.plot(h.MD, h.Z, lw=2, color="tab:blue", label="Z del pozo")
    for c in TOPS:
        ax.plot(h.MD, h[c], lw=1, label=c)
    ax.axvline(md_ps, color="red", ls="--")
    ax.set_ylabel("Z (ft)"); ax.set_title("Trayectoria vertical y topes de formación"); ax.legend(ncol=4, fontsize=8)

    ax = fig.add_subplot(gs[2, 0], sharex=ax)
    ax.plot(h.MD, h.TVT, color="tab:green", label="TVT (target)")
    ax.plot(h.MD, h.TVT_input, color="k", lw=2, label="TVT_input (conocido)")
    ax.axvline(md_ps, color="red", ls="--")
    ax.invert_yaxis(); ax.set_xlabel("MD (ft)"); ax.set_ylabel("TVT (ft)"); ax.legend()
    ax.set_title("Target: TVT vs MD")

    ax = fig.add_subplot(gs[:, 1])
    ax.plot(t.GR, t.TVT, color="red", lw=0.8, label="GR typewell")
    ax.plot(h.GR, h.TVT, color="k", lw=0.5, alpha=0.7, label="GR horizontal (proyectado en TVT)")
    for name, g in t.dropna(subset=["Geology"]).groupby("Geology", sort=False):
        ax.axhline(g.TVT.min(), color="gray", ls=":", lw=0.8)
        ax.text(205, g.TVT.min(), name, fontsize=7, va="center")
    ax.set_ylim(h.TVT.max() + 30, h.TVT.min() - 30); ax.set_xlim(0, 200)
    ax.set_xlabel("GR"); ax.set_ylabel("TVT (ft)"); ax.legend(fontsize=8)
    ax.set_title("GR proyectado en TVT vs typewell")

    fig.suptitle(f"Pozo {wid}", fontsize=14)
    fig.tight_layout()

plot_well(h, t, wid)

Lectura de la figura:

- La parte conocida (TVT_input) cubre el tramo curvo inicial; lo que hay que predecir es casi toda la sección lateral.
- El GR horizontal, cuando se proyecta en TVT, reproduce la firma del GR del typewell. Esa correlación es la información central del problema.
- Los topes (`ANCC`…`BUDA`) son la geología interpretada en coordenadas Z y **sólo existen en train**; en test no están.

## 3. Estadísticas por pozo (todo el set de entrenamiento)

In [ ]:
rows, wells = [], {}
for w in tqdm(train_ids):
    hw, tw = load_well(w)
    wells[w] = (hw, tw)
    ps = int(hw["TVT_input"].notna().sum())
    pred = hw.iloc[ps:]
    dx, dy = hw.X.iloc[-1] - hw.X.iloc[ps], hw.Y.iloc[-1] - hw.Y.iloc[ps]
    rows.append(dict(
        well=w, n=len(hw), ps=ps, n_pred=len(hw) - ps, frac_pred=1 - ps / len(hw),
        md_step=hw.MD.diff().median(),
        gr_nan=hw.GR.isna().mean(), gr_nan_pred=pred.GR.isna().mean(),
        gr_mean=hw.GR.mean(), gr_std=hw.GR.std(),
        tvt_min=hw.TVT.min(), tvt_max=hw.TVT.max(), tvt_range=hw.TVT.max() - hw.TVT.min(),
        tvt_ps=hw.TVT.iloc[ps], tvt_end=hw.TVT.iloc[-1],
        dtvt_pred_absmax=(pred.TVT - hw.TVT.iloc[ps]).abs().max(),
        dtvt_pred_rms=np.sqrt(((pred.TVT - hw.TVT.iloc[ps]) ** 2).mean()),
        z_range_pred=pred.Z.max() - pred.Z.min(),
        azimuth=np.degrees(np.arctan2(dx, dy)) % 360,
        lateral_len=np.hypot(dx, dy),
        x0=hw.X.iloc[0], y0=hw.Y.iloc[0], x_end=hw.X.iloc[-1], y_end=hw.Y.iloc[-1],
        tw_n=len(tw), tw_step=tw.TVT.diff().median(), tw_tvt_min=tw.TVT.min(), tw_tvt_max=tw.TVT.max(),
        tw_n_layers=tw.Geology.nunique(),
        tvt_in_tw=(hw.TVT.min() >= tw.TVT.min()) & (hw.TVT.max() <= tw.TVT.max()),
    ))
stats = pd.DataFrame(rows).set_index("well")
print("Columnas iguales en todos los pozos:", all(list(wells[w][0].columns) == list(h.columns) for w in train_ids))
stats.describe().T.round(2)

Tabla resumen: los puntos a retener de aquí son el tamaño de los pozos, qué fracción se predice, cuánto varía el target, cuántos NaN tiene el GR y la cobertura del typewell. Se visualizan a continuación.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
plots = [
    ("n", "Puntos por pozo (1 ft)"), ("ps", "Fila del Prediction Start"), ("frac_pred", "Fracción del pozo a predecir"),
    ("tvt_range", "Rango de TVT en el pozo (ft)"), ("gr_nan", "Fracción de GR NaN"), ("gr_mean", "GR medio del pozo"),
    ("tw_n", "Puntos del typewell"), ("lateral_len", "Longitud lateral predicha (ft)"),
]
for ax, (c, title) in zip(axes.ravel(), plots):
    sns.histplot(stats[c], ax=ax, bins=40, color="tab:blue")
    ax.set_title(title); ax.set_xlabel("")
fig.tight_layout()

In [ ]:
print("Paso de MD único:", stats.md_step.unique())
print("Paso del typewell (ft):", stats.tw_step.value_counts().to_dict())
print(f"Pozos cuyo TVT está totalmente cubierto por el rango del typewell: {stats.tvt_in_tw.mean():.1%}")
stats.loc[~stats.tvt_in_tw, ["tvt_min", "tvt_max", "tw_tvt_min", "tw_tvt_max"]].round(1)

## 4. Gamma ray: valores faltantes y distribución

El GR tiene una fracción grande de NaN (mediana cercana al 30 %). Importa saber si los huecos son puntos aislados (interpolables) o tramos largos (sin información).

In [ ]:
def nan_runs(s: pd.Series) -> np.ndarray:
    isna = s.isna().to_numpy().astype(int)
    if isna.sum() == 0:
        return np.array([], dtype=int)
    d = np.diff(np.concatenate([[0], isna, [0]]))
    return np.flatnonzero(d == -1) - np.flatnonzero(d == 1)

runs = np.concatenate([nan_runs(wells[w][0].GR) for w in train_ids])
run_tbl = pd.Series(runs).describe(percentiles=[0.5, 0.9, 0.99]).to_frame("longitud de huecos GR (ft)").T
display(run_tbl.round(1))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(runs[runs <= 20], bins=20, discrete=True, ax=axes[0]); axes[0].set_title("Longitud de huecos de GR (≤ 20 ft)"); axes[0].set_xlabel("ft")
sns.histplot(runs[runs > 20], bins=40, ax=axes[1], log_scale=(True, False)); axes[1].set_title("Huecos largos (> 20 ft)"); axes[1].set_xlabel("ft (log)")
sns.scatterplot(data=stats, x="gr_nan", y="gr_nan_pred", ax=axes[2], s=12)
axes[2].plot([0, 1], [0, 1], "r--", lw=1); axes[2].set_title("GR NaN: todo el pozo vs tramo a predecir")
fig.tight_layout()
print(f"Huecos de 1 ft: {np.mean(runs == 1):.1%} de los huecos; huecos > 50 ft: {np.sum(runs > 50)} en {len(train_ids)} pozos")

In [ ]:
all_gr_h = pd.concat([wells[w][0].GR for w in train_ids]).dropna()
all_gr_t = pd.concat([wells[w][1].GR for w in train_ids]).dropna()
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(all_gr_h.sample(300_000, random_state=0), ax=ax, label="GR pozo horizontal", fill=True)
sns.kdeplot(all_gr_t.sample(300_000, random_state=0), ax=ax, label="GR typewell", fill=True)
ax.set_xlim(0, 300); ax.set_title("Distribución global de GR"); ax.legend()
pd.DataFrame({"horizontal": all_gr_h.describe(), "typewell": all_gr_t.describe()}).round(1)

## 5. El target: TVT

Dos vistas útiles: (a) cuánto se aleja el TVT de su valor en el PS a lo largo del tramo a predecir, y (b) cómo es la pendiente local (dTVT/dMD), que es la que define si la geología sube, baja o es plana.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.histplot(stats.dtvt_pred_absmax, bins=40, ax=axes[0]); axes[0].set_title("Máx |TVT − TVT(PS)| en el tramo a predecir (ft)")
sns.histplot(stats.dtvt_pred_rms, bins=40, ax=axes[1]); axes[1].set_title("RMS de (TVT − TVT(PS)) por pozo (ft)")
sns.scatterplot(data=stats, x="z_range_pred", y="tvt_range", ax=axes[2], s=12)
axes[2].set_title("Rango de Z vs rango de TVT (tramo a predecir)"); axes[2].set_xlabel("rango Z (ft)"); axes[2].set_ylabel("rango TVT (ft)")
fig.tight_layout()
print("RMS de (TVT − TVT(PS)) agregado sobre todos los puntos a predecir:",
      round(np.sqrt(np.mean(np.concatenate([(wells[w][0].TVT.iloc[s.ps:] - s.tvt_ps).to_numpy() ** 2 for w, s in stats.iterrows()]))), 2), "ft")

El RMS respecto al valor en el PS es, por construcción, el error de una predicción "TVT constante". Sirve como referencia de la escala del problema: la variación del target dentro del tramo a predecir es de decenas de pies.

In [ ]:
# Pendiente local del target (ft de TVT por ft de MD), suavizada a 50 ft
slopes = []
for w in train_ids:
    hw = wells[w][0]
    ps = stats.loc[w, "ps"]
    s = hw.TVT.diff().rolling(50, center=True).mean().iloc[ps:]
    slopes.append(pd.DataFrame({"well": w, "slope": s.to_numpy(), "frac_md": np.linspace(0, 1, len(s))}))
slopes = pd.concat(slopes).dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(slopes.slope.clip(-0.3, 0.3), bins=80, ax=axes[0])
axes[0].set_title("dTVT/dMD (media móvil 50 ft) en el tramo a predecir"); axes[0].set_xlabel("ft/ft")
sns.lineplot(data=slopes.assign(bin=pd.cut(slopes.frac_md, 20, labels=False)).groupby("bin").slope.agg(["mean", "std"]).reset_index(),
             x="bin", y="mean", ax=axes[1], marker="o")
axes[1].set_title("Pendiente media del target vs posición relativa en el tramo"); axes[1].set_xlabel("posición en el tramo (0 = PS, 19 = fin)")
fig.tight_layout()
print(f"Fracción de puntos con |pendiente| < 0.01 ft/ft (geología casi plana): {(slopes.slope.abs() < 0.01).mean():.1%}")

In [ ]:
# Muestra aleatoria de curvas de target normalizadas al PS
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(12, 5))
for w in rng.choice(train_ids, 40, replace=False):
    hw = wells[w][0]; ps = stats.loc[w, "ps"]
    ax.plot(hw.MD.iloc[ps:] - hw.MD.iloc[ps], hw.TVT.iloc[ps:] - hw.TVT.iloc[ps], lw=0.8, alpha=0.7)
ax.axhline(0, color="k", lw=1); ax.invert_yaxis()
ax.set_xlabel("MD desde el PS (ft)"); ax.set_ylabel("TVT − TVT(PS) (ft)"); ax.set_title("40 pozos: evolución del target después del PS")

## 6. Geometría del pozo y relación con el target

TVT y Z están relacionados: si la geología fuera perfectamente plana, `TVT − Z` sería constante a lo largo del pozo. La desviación de esa constante es el "buzamiento" (dip) estructural que hay que capturar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for w in rng.choice(train_ids, 30, replace=False):
    hw = wells[w][0]; ps = stats.loc[w, "ps"]
    off = (hw.TVT + hw.Z)  # TVT crece hacia abajo, Z es negativo hacia abajo → TVT + Z ≈ cte si es plano
    axes[0].plot(hw.MD - hw.MD.iloc[ps], off - off.iloc[ps], lw=0.8, alpha=0.7)
axes[0].axvline(0, color="red", ls="--"); axes[0].set_xlabel("MD desde el PS (ft)"); axes[0].set_ylabel("(TVT + Z) − valor en PS (ft)")
axes[0].set_title("Desviación respecto a geología plana (30 pozos)")

ends = pd.DataFrame({"w": train_ids,
                     "offset_end": [(wells[w][0].TVT + wells[w][0].Z).iloc[-1] - (wells[w][0].TVT + wells[w][0].Z).iloc[stats.loc[w, 'ps']] for w in train_ids]})
sns.histplot(ends.offset_end, bins=50, ax=axes[1]); axes[1].set_title("Cambio de (TVT + Z) entre el PS y el final del pozo (ft)")
fig.tight_layout()

In [ ]:
fig = plt.figure(figsize=(14, 5))
ax0 = fig.add_subplot(1, 2, 1, projection="polar")
ax0.hist(np.radians(stats.azimuth), bins=36, color="tab:blue")
ax0.set_theta_zero_location("N"); ax0.set_theta_direction(-1); ax0.set_title("Azimut del tramo lateral (desde PS al final)")
ax1 = fig.add_subplot(1, 2, 2)
sns.scatterplot(data=stats, x="azimuth", y=ends.set_index("w").offset_end, ax=ax1, s=12)
ax1.axhline(0, color="k", lw=1)
ax1.set_xlabel("azimut (°)"); ax1.set_ylabel("cambio de TVT + Z (ft)"); ax1.set_title("Dip aparente vs azimut")
fig.tight_layout()

## 7. Distribución espacial de los pozos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sc = axes[0].scatter(stats.x0, stats.y0, c=stats.tvt_ps, s=10, cmap="viridis")
for w in train_ids:
    hw = wells[w][0]
    axes[0].plot([hw.X.iloc[0], hw.X.iloc[-1]], [hw.Y.iloc[0], hw.Y.iloc[-1]], color="gray", lw=0.4, alpha=0.6)
axes[0].scatter(stats.loc[test_ids, "x0"], stats.loc[test_ids, "y0"], s=80, facecolors="none", edgecolors="red", label="ids de test")
plt.colorbar(sc, ax=axes[0], label="TVT en el PS (ft)"); axes[0].set_title("Mapa de pozos (cabeza → fin) coloreado por TVT en el PS"); axes[0].legend()
axes[0].set_xlabel("X"); axes[0].set_ylabel("Y"); axes[0].set_aspect("equal")

sc = axes[1].scatter(stats.x0, stats.y0, c=ends.set_index("w").offset_end, s=12, cmap="coolwarm", vmin=-150, vmax=150)
plt.colorbar(sc, ax=axes[1], label="cambio de TVT + Z (ft)"); axes[1].set_title("Dip aparente por pozo"); axes[1].set_aspect("equal")
fig.tight_layout()

In [ ]:
from scipy.spatial import cKDTree
mid = np.c_[(stats.x0 + stats.x_end) / 2, (stats.y0 + stats.y_end) / 2]
d, _ = cKDTree(mid).query(mid, k=2)
stats["nn_dist"] = d[:, 1]
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(stats.nn_dist, bins=50, ax=ax); ax.set_title("Distancia al pozo vecino más cercano (ft, entre puntos medios)")
stats.nn_dist.describe().round(0).to_frame().T

## 8. Formaciones geológicas

En train tenemos los topes en Z (`ANCC`…`BUDA`) por cada punto del pozo. Comparando `Z` con los topes se obtiene en qué formación está cada punto. En el typewell, la columna `Geology` etiqueta capas en la escala de TVT.

In [ ]:
def formation_at(hw: pd.DataFrame) -> pd.Series:
    # Las formaciones están ordenadas de arriba (ANCC) a abajo (BUDA); Z negativo = más profundo
    tops = hw[TOPS].to_numpy()
    z = hw.Z.to_numpy()[:, None]
    below = (z <= tops).sum(axis=1)  # cuántos topes están por encima del punto
    labels = np.array(["sobre_" + TOPS[0]] + TOPS)
    return pd.Series(labels[below], index=hw.index)

fm = pd.concat([formation_at(wells[w][0]).iloc[stats.loc[w, "ps"]:] for w in train_ids])
fm_frac = fm.value_counts(normalize=True).rename("fracción de puntos a predecir")
display(fm_frac.round(3).to_frame())

thick = pd.DataFrame({f"{a}→{b}": pd.concat([(wells[w][0][a] - wells[w][0][b]) for w in train_ids]) for a, b in zip(TOPS[:-1], TOPS[1:])})
fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=thick.sample(100_000, random_state=0), ax=ax, showfliers=False); ax.set_ylabel("espesor (ft)"); ax.set_title("Espesor entre topes consecutivos (todos los puntos)")

In [ ]:
geo_counts = pd.concat([wells[w][1].Geology.dropna().drop_duplicates() for w in train_ids]).value_counts()
geo_counts.rename("nº de typewells donde aparece la capa").to_frame().head(20)

## 9. Typewell vs pozo horizontal: cómo se parece el GR

El pptx sugiere que el GR del pozo horizontal (antes del PS) correlaciona mejor con el GR del mismo pozo después del PS que con el GR del typewell. Comparamos, por pozo, el GR horizontal promediado en celdas de TVT contra el typewell en el mismo rango.

In [ ]:
def gr_on_tvt_grid(hw, tw, step=1.0):
    lo, hi = hw.TVT.min(), hw.TVT.max()
    grid = np.arange(np.floor(lo), np.ceil(hi) + step, step)
    gh = hw.dropna(subset=["GR"]).groupby(pd.cut(hw.TVT, grid), observed=True).GR.mean()
    gt = tw.groupby(pd.cut(tw.TVT, grid), observed=True).GR.mean()
    both = pd.concat([gh.rename("h"), gt.rename("t")], axis=1).dropna()
    return both

corrs = pd.Series({w: gr_on_tvt_grid(*wells[w]).corr().iloc[0, 1] for w in train_ids}, name="corr(GR_h, GR_tw) en TVT")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(corrs, bins=40, ax=axes[0]); axes[0].set_title("Correlación GR horizontal vs typewell (celdas de 1 ft de TVT)")
best, worst = corrs.idxmax(), corrs.idxmin()
for w, c in [(best, "tab:green"), (worst, "tab:red")]:
    b = gr_on_tvt_grid(*wells[w])
    axes[1].scatter(b.t, b.h, s=4, color=c, alpha=0.5, label=f"{w} (r={corrs[w]:.2f})")
axes[1].plot([0, 250], [0, 250], "k--", lw=1); axes[1].set_xlabel("GR typewell"); axes[1].set_ylabel("GR horizontal"); axes[1].legend()
axes[1].set_title("Mejor y peor pozo")
fig.tight_layout()
corrs.describe().round(3).to_frame().T

In [ ]:
plot_well(*wells[worst], worst)

## 10. Resumen de hallazgos

Observaciones de esta primera pasada (los números salen de las celdas de arriba):

- **Datos**: 773 pozos de entrenamiento con columnas idénticas y paso de 1 ft en MD. Los 3 pozos de `test/` son copias de train sin target, así que la validación local tiene que hacerse separando pozos de train. Hay vecinos a menos de 50 ft (mediana ≈ 470 ft), por lo que conviene una partición espacial para no filtrar información entre pozos casi gemelos.
- **Tramo a predecir**: es la mayor parte del pozo (mediana ≈ 74 %, ~4.900 puntos por pozo). Es decir, se predice una serie larga a partir de un tramo conocido corto.
- **GR**: ~30 % de NaN, pero el 63 % de los huecos son de 1 ft y el 99 % de ≤ 9 ft (interpolables); sólo 70 huecos > 50 ft en todo el set. La distribución de GR del pozo horizontal y del typewell es similar pero no idéntica (typewell con más dispersión).
- **Target**: alejarse del valor del PS tiene un RMS global de ≈ 16 ft, y el 45 % de los puntos tienen pendiente casi nula. Las curvas de TVT después del PS son suaves, sin saltos.
- **Dip regional**: `TVT + Z` cambia de forma casi lineal con el MD y su signo depende del azimut: pozos hacia el NO (≈ 300-360°) acumulan +100…+300 ft y pozos hacia el SE (≈ 130-190°) −100…−300 ft. Hay una pendiente estructural regional consistente; la geometría del pozo (XYZ + azimut) por sí sola explica una parte grande del target.
- **Typewell**: paso de 0.5 ft en la mayoría (también 0.25 y 1 ft). Cubre el rango de TVT del pozo en el 98 % de los casos; 13 pozos no quedan cubiertos, 11 de ellos comparten el mismo typewell (TVT 11.782-12.285). El GR horizontal proyectado en TVT correlaciona con el del typewell con mediana 0.84, pero hay una cola con correlación < 0.6 que merece revisarse pozo por pozo.
- **Formaciones**: los topes sólo existen en train. El 87.5 % de los puntos a predecir caen en EGFDL y el 8 % en EGFDU; en el typewell aparecen además capas locales (MNSS, OLMOS, LTHL…) en una minoría de pozos.